# Obtain
Obtain the challenge data set and the weather data

In [ ]:
import pandas as pd

df = pd.read_csv("Training_data_uhi_index_2025-02-18.csv")

In [ ]:
display(df)

,Longitude,Latitude,datetime,UHI Index
0,-73.909167,40.813107,24-07-2021 15:53,1.030289
1,-73.909187,40.813045,24-07-2021 15:53,1.030289
2,-73.909215,40.812978,24-07-2021 15:53,1.023798
3,-73.909242,40.812908,24-07-2021 15:53,1.023798
4,-73.909257,40.812845,24-07-2021 15:53,1.021634
...,...,...,...,...
11224,-73.957050,40.790333,24-07-2021 15:57,0.972470
11225,-73.957063,40.790308,24-07-2021 15:57,0.972470
11226,-73.957093,40.790270,24-07-2021 15:57,0.981124
11227,-73.957112,40.790253,24-07-2021 15:59,0.981245


In [ ]:
df.shape

(11229, 4)

In [ ]:
# prompt: # prompt: # prompt: assign labels "area" to each Latitude and Longitude using import requests
# # # # Replace with your target coordinates
# # # lat = 40.8200
# # # lon = -73.9500
# # # # Nominatim endpoint for reverse geocoding
# # # url = "https://nominatim.openstreetmap.org/reverse"
# # # params = {
# # #     'lat': lat,
# # #     'lon': lon,
# # #     'format': 'jsonv2'
# # # }
# # # # Nominatim requires a valid User-Agent header
# # # headers = {
# # #     'User-Agent': 'YourAppName/1.0 (your.email@example.com)'
# # # }
# # # response = requests.get(url, params=params, headers=headers)
# # # data = response.json()
# # # # Print out the complete address information
# # # print("Address Data:")
# # # print(data.get("address", {}))
# # # # Check common fields for borough information
# # # address = data.get("address", {})
# # # borough = address.get("borough") or address.get("city_district") or address.get("suburb")
# # # if borough:
# # #     print(f"\nThe coordinate is in: {borough}")
# # # else:
# # #     print("\nBorough information not found in the reverse geocoding data.")
# # # Implement tqdm as well for progress
# # Rework the code to do in batch of size 128 with 0.25 second sleep to rate limit the API. After every batch finishes, download that batch's file. Show progress within the batch and progress for each batch. Also allow me to define a start idx

import pandas as pd
import requests
from time import sleep
from tqdm import tqdm
import os

# Replace with your target coordinates
# lat = 40.8200
# lon = -73.9500

# Nominatim endpoint for reverse geocoding
url = "https://nominatim.openstreetmap.org/reverse"
headers = {
    'User-Agent': 'YourAppName/1.0 (your.email@example.com)' # Replace with your actual User-Agent
}

df = pd.read_csv("Training_data_uhi_index_2025-02-18.csv")

BATCH_SIZE = 128
SLEEP_TIME = 0.25
START_IDX = 11 # You can change this to start from a different index


def get_area(lat, lon):
    params = {
        'lat': lat,
        'lon': lon,
        'format': 'jsonv2'
    }
    try:
        response = requests.get(url, params=params, headers=headers)
        response.raise_for_status() # Raise an exception for bad status codes
        data = response.json()
        address = data.get("address", {})
        borough = address.get("borough") or address.get("city_district") or address.get("suburb") or address.get("county") or address.get("state")
        return borough
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for coordinates ({lat}, {lon}): {e}")
        return None


areas = []
for i in tqdm(range(START_IDX, len(df), BATCH_SIZE), desc="Processing Batches"):
    batch_df = df.iloc[i:min(i + BATCH_SIZE, len(df))]
    batch_areas = []
    for index, row in tqdm(batch_df.iterrows(), total=len(batch_df), desc=f"Batch {i // BATCH_SIZE + 1}"):
        area = get_area(row["Latitude"], row["Longitude"])
        batch_areas.append(area)
        sleep(SLEEP_TIME)
    areas.extend(batch_areas)

    # Save the batch's results to a file (optional)
    batch_df['area'] = batch_areas
    batch_df.to_csv(f"batch_{i // BATCH_SIZE + 1}.csv", index=False)

# add the area to the main df
df['area'] = areas
df.to_csv("output_with_area.csv", index=False)
print("Finished processing all coordinates.")


Batch 1:  44%|████▍     | 56/128 [01:00<01:16,  1.06s/it]

Error fetching data for coordinates (40.80982833, -73.90739833): 403 Client Error: Forbidden for url: https://nominatim.openstreetmap.org/reverse?lat=40.80982833&lon=-73.90739833&format=jsonv2



Batch 1:  45%|████▍     | 57/128 [01:01<01:19,  1.11s/it]

Error fetching data for coordinates (40.80977667, -73.90741167): 403 Client Error: Forbidden for url: https://nominatim.openstreetmap.org/reverse?lat=40.80977667&lon=-73.90741167&format=jsonv2



Batch 1:  45%|████▌     | 58/128 [01:04<01:57,  1.68s/it]

Error fetching data for coordinates (40.80964333, -73.907465): 403 Client Error: Forbidden for url: https://nominatim.openstreetmap.org/reverse?lat=40.80964333&lon=-73.907465&format=jsonv2


... (96 outputs left out)


Batch 4:  52%|█████▏    | 66/128 [02:20<03:06,  3.01s/it]

Error fetching data for coordinates (40.80692833, -73.92745333): 403 Client Error: Forbidden for url: https://nominatim.openstreetmap.org/reverse?lat=40.80692833&lon=-73.92745333&format=jsonv2



Processing Batches:   3%|▎         | 3/88 [10:16<4:50:57, 205.38s/it]


KeyboardInterrupt: 

In [ ]:
# prompt: # prompt: assign labels "area" to each Latitude and Longitude using import requests
# # # Replace with your target coordinates
# # lat = 40.8200
# # lon = -73.9500
# # # Nominatim endpoint for reverse geocoding
# # url = "https://nominatim.openstreetmap.org/reverse"
# # params = {
# #     'lat': lat,
# #     'lon': lon,
# #     'format': 'jsonv2'
# # }
# # # Nominatim requires a valid User-Agent header
# # headers = {
# #     'User-Agent': 'YourAppName/1.0 (your.email@example.com)'
# # }
# # response = requests.get(url, params=params, headers=headers)
# # data = response.json()
# # # Print out the complete address information
# # print("Address Data:")
# # print(data.get("address", {}))
# # # Check common fields for borough information
# # address = data.get("address", {})
# # borough = address.get("borough") or address.get("city_district") or address.get("suburb")
# # if borough:
# #     print(f"\nThe coordinate is in: {borough}")
# # else:
# #     print("\nBorough information not found in the reverse geocoding data.")
# # Implement tqdm as well for progress
# Rework the code to do in batch of size 128 with 0.25 second sleep to rate limit the API. After every batch finishes, download that batch's file. Show progress within the batch and progress for each batch

import pandas as pd
import requests
from tqdm import tqdm
import time
import os

# Assuming df is already defined from the previous code block
# df = pd.read_csv("Training_data_uhi_index_2025-02-18.csv")

# Parameters
BATCH_SIZE = 128
SLEEP_TIME = 0.25
OUTPUT_DIR = "geocoded_batches"  # Directory to save batch files

# Create the output directory if it doesn't exist
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)


def geocode_coordinates(lat, lon):
    url = "https://nominatim.openstreetmap.org/reverse"
    params = {
        'lat': lat,
        'lon': lon,
        'format': 'jsonv2'
    }
    headers = {
        'User-Agent': 'YourAppName/1.0 (your.email@example.com)'  # Replace with your User-Agent
    }
    try:
      response = requests.get(url, params=params, headers=headers)
      response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
      data = response.json()
      address = data.get("address", {})
      borough = address.get("borough") or address.get("city_district") or address.get("suburb") or address.get("town") or address.get("city")
      return borough
    except requests.exceptions.RequestException as e:
      print(f"Error geocoding coordinates {lat}, {lon}: {e}")
      return None


# Process in batches
num_batches = (len(df) + BATCH_SIZE - 1) // BATCH_SIZE

for i in range(num_batches):
    start_idx = i * BATCH_SIZE
    end_idx = min((i + 1) * BATCH_SIZE, len(df))
    batch_df = df[start_idx:end_idx].copy()

    areas = []
    for index, row in tqdm(batch_df.iterrows(), total=len(batch_df), desc=f"Batch {i + 1}/{num_batches}"):
        latitude = row['Latitude']
        longitude = row['Longitude']
        area = geocode_coordinates(latitude, longitude)
        areas.append(area)
        time.sleep(SLEEP_TIME)

    batch_df['area'] = areas
    batch_filename = os.path.join(OUTPUT_DIR, f"geocoded_batch_{i + 1}.csv")
    batch_df.to_csv(batch_filename, index=False)
    print(f"Batch {i + 1} completed and saved to {batch_filename}")

print("Geocoding completed for all batches.")


Batch 1/88: 100%|██████████| 128/128 [02:08<00:00,  1.00s/it]


Batch 1 completed and saved to geocoded_batches/geocoded_batch_1.csv


Batch 2/88: 100%|██████████| 128/128 [02:08<00:00,  1.00s/it]


Batch 2 completed and saved to geocoded_batches/geocoded_batch_2.csv


Batch 3/88: 100%|██████████| 128/128 [02:07<00:00,  1.00it/s]


Batch 3 completed and saved to geocoded_batches/geocoded_batch_3.csv


... (47 outputs left out)

Batch 27 completed and saved to geocoded_batches/geocoded_batch_27.csv


Batch 28/88: 100%|██████████| 128/128 [02:08<00:00,  1.00s/it]


Batch 28 completed and saved to geocoded_batches/geocoded_batch_28.csv


Batch 29/88:   5%|▍         | 6/128 [00:06<02:02,  1.01s/it]

In [ ]:
# prompt: read the Bronx and Manhattan sheets from NY_Mesonet_Weather.xlsx, create a label whether it is Bronx or Manhattan and filter only to 3 to 4PM and group by date (make sure to clean the string EDT first before converting it to date time) aggregate average and join the weather related data on to the df based on the label

import pandas as pd

# Load the Excel file
xls = pd.ExcelFile('NY_Mesonet_Weather.xlsx')

# Read the Bronx and Manhattan sheets
bronx_df = pd.read_excel(xls, 'Bronx')
manhattan_df = pd.read_excel(xls, 'Manhattan')

# Add a label column
bronx_df['label'] = 'Bronx'
manhattan_df['label'] = 'Manhattan'

# Combine the dataframes
combined_df = pd.concat([bronx_df, manhattan_df])


# Clean the 'Time' column and convert to datetime
combined_df['Date Time'] = combined_df['Date / Time'].astype(str).str.replace('EDT', '')
combined_df['Date'] = pd.to_datetime(combined_df['Date Time'])

# Filter the dataframe for time between 3 PM and 4 PM
combined_df = combined_df[(combined_df['Date'].dt.hour >= 15) & (combined_df['Date'].dt.hour < 16)]

# Group the dataframe by date and calculate the average for relevant columns
combined_df['Date'] = combined_df['Date'].dt.date
weather_data = combined_df.groupby(['Date','label'])[['Air Temp at Surface [degC]', 'Relative Humidity [percent]', 'Avg Wind Speed [m/s]', 'Wind Direction [degrees]', 'Solar Flux [W/m^2]']].mean().reset_index()
weather_data
# Example join operation, replace with your actual join key and weather data columns
#df = df.merge(weather_data, on='Date', how='left')


In [ ]:
# prompt: join the areas from the df to label from weather data to enrich df with weather data

import pandas as pd
# Assuming 'Date' column exists in both dataframes and represents the date
df['Date'] = pd.to_datetime(df['datetime']).dt.date

# Merge the dataframes based on the 'area' and 'Date' columns
df = df.merge(weather_data, left_on=['Date', 'area'], right_on=['Date', 'label'], how='left')

# Display the updated DataFrame
display(df)


In [ ]:
# prompt: download df

import pandas as pd
from google.colab import files

# Assuming df is your DataFrame from the previous code (replace with your actual DataFrame)

# Convert the 'Date' column to datetime objects if it's not already
df['Date'] = pd.to_datetime(df['Date'])

# Create a CSV file from the DataFrame
df.to_csv('enriched_data.csv', index=False)

# Download the CSV file
files.download('enriched_data.csv')


# Scrub

In [ ]:
# prompt: order by datetime

import pandas as pd
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values(by='datetime')
display(df)


In [ ]:
# prompt: generate a feature called minute which represents the minute portion of the datetime

df['minute'] = df['datetime'].dt.minute


In [ ]:
# prompt: average minute per Latitude and Longitude, corrected to the nearest integer

import pandas as pd
import numpy as np

# Assuming df is already loaded and processed as in the previous code

# Group by Latitude and Longitude and calculate the average minute, rounding to the nearest integer
average_minute_by_location = df.groupby(['Latitude', 'Longitude'])['minute'].mean().round().astype(int)

# Display the result
display(average_minute_by_location)

# If you want to merge this back into the original DataFrame:
df = df.merge(average_minute_by_location.rename('average_minute'), left_on=['Latitude', 'Longitude'], right_index=True)

# Display the updated DataFrame
display(df)


In [ ]:
# prompt: delete minute and rename average_minute to minute

import pandas as pd
import numpy as np
# # Obtain
# Obtain the challenge data set and the weather data

df = pd.read_csv("Training_data_uhi_index_2025-02-18.csv")
display(df)
df.shape
# # Scrub

df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values(by='datetime')
display(df)


df['minute'] = df['datetime'].dt.minute



# Assuming df is already loaded and processed as in the previous code

# Group by Latitude and Longitude and calculate the average minute, rounding to the nearest integer
average_minute_by_location = df.groupby(['Latitude', 'Longitude'])['minute'].mean().round().astype(int)

# Display the result
display(average_minute_by_location)

# If you want to merge this back into the original DataFrame:
df = df.merge(average_minute_by_location.rename('average_minute'), left_on=['Latitude', 'Longitude'], right_index=True)

# Display the updated DataFrame
display(df)

# Delete the 'minute' column
del df['minute']

# Rename the 'average_minute' column to 'minute'
df = df.rename(columns={'average_minute': 'minute'})

# Display the final DataFrame
display(df)


In [ ]:
# prompt: create a dataframe that has columns minute and average UHI for each of the minute

# Assuming df is already defined and processed as in the previous code

# Calculate the average UHI for each minute
df_minute_avg = df.groupby('minute')['UHI Index'].mean().reset_index()
df_minute_avg = df_minute_avg.rename(columns={'UHI Index': 'average_UHI'})

display(df_minute_avg)


In [ ]:
# prompt: join average UHI to df based on average minute from left table to minute on right table

import pandas as pd
# Merge the average UHI back into the original DataFrame based on the 'minute' column
df = pd.merge(df, df_minute_avg, on='minute', how='left')

# Display the updated DataFrame
display(df)


In [ ]:
# prompt: for each lat long, get the average UHI Index (target feature) and the average of average minute based UHI

# Assuming df is already defined and processed as in the previous code

# Group by Latitude, Longitude and calculate the average UHI index and average of average minute based UHI
result = df.groupby(['Latitude', 'Longitude']).agg(
    average_UHI_index=('UHI Index', 'mean'),
    average_of_average_minute_UHI=('average_UHI', 'mean')
)

# Display the result
display(result)


# Explore

In [ ]:
min(df.datetime), max(df.datetime)

In [ ]:
import pandas as pd
from datetime import datetime

def analyze_datetime_granularity_df(df, datetime_col='datetime'):
    """
    Analyze the granularity of datetime values in a DataFrame

    Parameters:
    df: pandas DataFrame
    datetime_col: name of the datetime column (default='datetime')
    """
    # Ensure datetime column is datetime type
    if not pd.api.types.is_datetime64_any_dtype(df[datetime_col]):
        df[datetime_col] = pd.to_datetime(df[datetime_col])

    # Analyze time differences
    time_diffs = df[datetime_col].diff().dropna()

    analysis = {
        'min_difference': time_diffs.min(),
        'max_difference': time_diffs.max(),
        'unique_times': df[datetime_col].nunique(),
        'total_records': len(df),
        'has_seconds': any('S' in str(diff) for diff in time_diffs),
        'has_milliseconds': any('ms' in str(diff) for diff in time_diffs),
        'time_range': df[datetime_col].max() - df[datetime_col].min(),
        'sample_values': df[datetime_col].head().tolist()
    }

    return analysis

def format_time_analysis(analysis):
    """
    Format the analysis results into a readable string
    """
    return f"""Datetime Granularity Analysis:

- Smallest time difference: {analysis['min_difference']}
- Largest time difference: {analysis['max_difference']}
- Number of unique timestamps: {analysis['unique_times']}
- Total records: {analysis['total_records']}
- Contains seconds: {analysis['has_seconds']}
- Contains milliseconds: {analysis['has_milliseconds']}
- Total time range: {analysis['time_range']}
- Sample values: {[str(x) for x in analysis['sample_values']]}"""

# If your datetime column is named 'datetime':
results = analyze_datetime_granularity_df(df)

# Or if your datetime column has a different name:
results = analyze_datetime_granularity_df(df, datetime_col='datetime')

# Print the formatted results
print(format_time_analysis(results))

In [ ]:
# prompt: group by longitude latitude for count of rows, order by count

import pandas as pd

# Assuming your DataFrame is named 'df'
grouped = df.groupby(['Longitude', 'Latitude']).size().reset_index(name='counts')
ordered_grouped = grouped.sort_values('counts', ascending=False)
ordered_grouped


^ This confirms that each longitude and latitude occurs only once.

In [ ]:
# prompt: # prompt: # prompt: group by date time get count
# # import pandas as pd
# # # Assuming your DataFrame is named 'df' and 'datetime' column exists
# # # and is already in datetime format as per your previous code.
# # # Group by date and get the count
# # df['date'] = df['datetime'].dt.date  # Extract the date part
# # date_counts = df.groupby('date').size().reset_index(name='count')
# # date_counts
# # group by date time
# # Group by datetime and get the count
# datetime_counts = df.groupby('datetime').size().reset_index(name='count')
# datetime_counts
# Get count, average Longitude, average Latitude, average UHI Index and median UHI Index

import pandas as pd

# Assuming your DataFrame is named 'df' and 'datetime' column exists
# and is already in datetime format.

# Group by datetime and get the count, along with other aggregated values
datetime_stats = df.groupby('datetime').agg(
    count=('UHI Index', 'count'),
    mean_longitude=('Longitude', 'mean'),
    mean_latitude=('Latitude', 'mean'),
    mean_uhi=('UHI Index', 'mean'),
    median_uhi=('UHI Index', 'median')
).reset_index()

datetime_stats


In [ ]:
# prompt: Distribution of UHI Index

import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'uhi_index' is the column containing the UHI index values
plt.figure(figsize=(10, 6))
sns.histplot(df['UHI Index'], kde=True)  # kde=True adds a kernel density estimate
plt.title('Distribution of UHI Index')
plt.xlabel('UHI Index')
plt.ylabel('Frequency')
plt.show()


# Boxplot
plt.figure(figsize=(8, 6))
sns.boxplot(y=df['UHI Index'])
plt.title('Boxplot of UHI Index')
plt.ylabel('UHI Index')
plt.show()


In [ ]:
# prompt: examine outliers by iqr method

import pandas as pd

# Calculate Q1, Q3, and IQR
Q1 = df['UHI Index'].quantile(0.25)
Q3 = df['UHI Index'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify outliers
outliers = df[(df['UHI Index'] < lower_bound) | (df['UHI Index'] > upper_bound)]

# Print or display outliers
print("Outliers (IQR Method):")
outliers

# You can also visualize the outliers using a boxplot (already shown in previous code)
# or by creating a scatter plot highlighting outliers


In [ ]:
# prompt: remove outliers

# Remove outliers
df_no_outliers = df[(df['UHI Index'] >= lower_bound) & (df['UHI Index'] <= upper_bound)]

# Print or display the DataFrame without outliers
print("DataFrame without outliers:")
df_no_outliers


In [ ]:
# prompt: Distribution of UHI Index

import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'uhi_index' is the column containing the UHI index values
plt.figure(figsize=(10, 6))
sns.histplot(df_no_outliers['UHI Index'], kde=True)  # kde=True adds a kernel density estimate
plt.title('Distribution of UHI Index')
plt.xlabel('UHI Index')
plt.ylabel('Frequency')
plt.show()


# Boxplot
plt.figure(figsize=(8, 6))
sns.boxplot(y=df_no_outliers['UHI Index'])
plt.title('Boxplot of UHI Index')
plt.ylabel('UHI Index')
plt.show()


In [ ]:
# prompt: group Longitude Latitude aggregate average UHI Index

import pandas as pd

# Assuming 'df_no_outliers' is your DataFrame without outliers
# Group by 'Longitude' and 'Latitude' and calculate the average 'UHI Index'
grouped_df = df_no_outliers.groupby(['Longitude', 'Latitude'])['UHI Index'].mean().reset_index()

# Display the grouped DataFrame
display(grouped_df)


In [ ]:
df.shape[0] - grouped_df.shape[0]

In [ ]:
# prompt: plot distribution of UHI Index again

# Assuming 'df_no_outliers' is your DataFrame without outliers from the previous code

import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'uhi_index' is the column containing the UHI index values
plt.figure(figsize=(10, 6))
sns.histplot(grouped_df['UHI Index'], kde=True)  # kde=True adds a kernel density estimate
plt.title('Distribution of UHI Index (Average)')
plt.xlabel('UHI Index')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# prompt: distribution of minute

import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'df' is your DataFrame and 'minute' column exists

plt.figure(figsize=(10, 6))
sns.histplot(df['minute'], kde=True, bins=60)  # Use bins=60 for minutes
plt.title('Distribution of Minute')
plt.xlabel('Minute')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# prompt: visualise average_UHI

import matplotlib.pyplot as plt
# Assuming 'df_minute_avg' is already created as in the previous code

plt.figure(figsize=(10, 6))
sns.lineplot(x='minute', y='average_UHI', data=df_minute_avg)
plt.title('Average UHI Index over Minutes')
plt.xlabel('Minute')
plt.ylabel('Average UHI Index')
plt.show()


In [ ]:
# prompt: visualise average_UHI by lat long

import matplotlib.pyplot as plt

# Assuming 'grouped_df' is your DataFrame with 'Longitude', 'Latitude', and 'UHI Index' columns
plt.figure(figsize=(10, 8))  # Adjust figure size as needed
plt.scatter(grouped_df['Longitude'], grouped_df['Latitude'], c=grouped_df['UHI Index'], cmap='viridis')
plt.colorbar(label='Average UHI Index')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Average UHI Index by Latitude and Longitude')
plt.show()


# Model

In [ ]:
result_df = result.reset_index()
display(result_df)

In [ ]:
# prompt: cross fold validation of linear regression, ridge regression, lasso regression, elastic net, naive bayes regression, decision tree, random forest, light gbm, xgboost, svr, and knn regression and report the results sorted in descending order of R2 score.

import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
import numpy as np

# Assuming 'grouped_df' is your preprocessed DataFrame
# Replace 'UHI Index' with the actual target variable column name if different
X = result_df.drop('average_UHI_index', axis=1)
y = result_df['average_UHI_index']

# Create a dictionary of models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'Elastic Net': ElasticNet(),
    'Naive Bayes Regression': GaussianNB(),  # Note: Naive Bayes is a classification algorithm, use with caution for regression
    'Decision Tree': DecisionTreeRegressor(),
    'Random Forest': RandomForestRegressor(),
    'LightGBM': LGBMRegressor(),
    'XGBoost': XGBRegressor(),
    'SVR': SVR(),
    'KNN Regression': KNeighborsRegressor()
}

# Perform cross-validation and store results
results = {}
kf = KFold(n_splits=5, shuffle=True, random_state=42) # Example 5-fold CV

for name, model in models.items():
  try:
    cv_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')
    results[name] = np.mean(cv_scores)
  except Exception as e:
    # print(f"Error with {name}: {e}")
    results[name] = np.nan

# Sort results by R-squared score
sorted_results = dict(sorted(results.items(), key=lambda item: item[1], reverse=True))

# Print the sorted results
for name, score in sorted_results.items():
  print(f"{name}: {score:.4f}")


In [ ]:
X.columns

# Interpret

In [ ]:
# prompt: interpret the model for insights on the feature importance

import pandas as pd
import matplotlib.pyplot as plt
# Assuming 'X' is your feature matrix and 'models' is your dictionary of trained models.
# You'll need to adapt this based on the specific model you choose to interpret.

importances = {}

# Example: Get feature importances from a RandomForestRegressor
model = models['Random Forest']  # Or whichever model you want to analyze
if isinstance(model, RandomForestRegressor):
    model.fit(X,y)
    importances = model.feature_importances_
    feature_names = X.columns
    feature_importance_df = pd.DataFrame({'feature':feature_names, 'importance': importances})
    print(feature_importance_df.sort_values('importance', ascending=False))

# Example using permutation importance for any model:
from sklearn.inspection import permutation_importance
result = permutation_importance(model, X, y, n_repeats=10, random_state=42, n_jobs=2)
sorted_idx = result.importances_mean.argsort()

fig, ax = plt.subplots()
ax.boxplot(result.importances[sorted_idx].T, vert=False, labels=X.columns[sorted_idx])
ax.set_title("Permutation Importances (test set)")
fig.tight_layout()
plt.show()


#More detailed explanation of the feature importance scores based on the selected model.
# For tree-based models, higher values indicate stronger predictive power.
# For linear models, the coefficients show the effect of each feature on the target.
# If using permutation importance, the ranking reflects how much the model's accuracy decreases when the feature is randomly shuffled.


In [ ]:
import matplotlib.pyplot as plt
# Assuming 'grouped_df' is your DataFrame with 'Latitude' and 'UHI Index' columns
plt.figure(figsize=(10, 6))
sns.regplot(x='Latitude', y='UHI Index', data=grouped_df, scatter_kws={"color": "blue"}, line_kws={"color": "red"})
plt.xlabel('Latitude')
plt.ylabel('UHI Index')
plt.title('UHI Index vs. Latitude')
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
# Assuming 'grouped_df' is your DataFrame with 'Latitude' and 'UHI Index' columns
plt.figure(figsize=(10, 6))
sns.regplot(x='Longitude', y='UHI Index', data=grouped_df, scatter_kws={"color": "blue"}, line_kws={"color": "red"})
plt.xlabel('Latitude')
plt.ylabel('UHI Index')
plt.title('UHI Index vs. Latitude')
plt.grid(True)
plt.show()


In [ ]:
# prompt: import matplotlib.pyplot as plt
# # Assuming 'grouped_df' is your DataFrame with 'Latitude' and 'UHI Index' columns
# plt.figure(figsize=(10, 6))
# sns.regplot(x='Longitude', y='UHI Index', data=grouped_df, scatter_kws={"color": "blue"}, line_kws={"color": "red"})
# plt.xlabel('Latitude')
# plt.ylabel('UHI Index')
# plt.title('UHI Index vs. Latitude')
# plt.grid(True)
# plt.show()
# Like above plot correlation of UHI Index and average_of_average_minute_UHI

import matplotlib.pyplot as plt
# Assuming 'result_df' is your DataFrame with 'average_UHI_index' and 'average_of_average_minute_UHI' columns
plt.figure(figsize=(10, 6))
sns.regplot(x='average_of_average_minute_UHI', y='average_UHI_index', data=result_df, scatter_kws={"color": "blue"}, line_kws={"color": "red"})
plt.xlabel('Average of Average Minute UHI')
plt.ylabel('Average UHI Index')
plt.title('Average UHI Index vs. Average of Average Minute UHI')
plt.grid(True)
plt.show()


# Submission

In [ ]:
# prompt: retrain the best performing model on all the data (without cross-fold validation)

# Assuming 'X' and 'y' are your features and target variable from the grouped_df
# and 'sorted_results' contains the cross-validation results.

# Retrain the best performing model on the entire dataset
best_model_name = list(sorted_results.keys())[1] # Get the name of the best model
best_model = models[best_model_name]

best_model.fit(X, y) # Fit on the entire dataset

# Now 'best_model' is trained on all the data and ready for predictions on new data.
# You can save this model for later use if needed.
import joblib
joblib.dump(best_model, f'{best_model_name}_final_model.pkl')


In [ ]:
# prompt: use the model to predict X and use y as y_true and report the r2

from sklearn.metrics import r2_score

# Assuming 'best_model' is already trained and 'X' contains the features for prediction
# and 'y' is your true target values

y_pred = best_model.predict(X)
r2 = r2_score(y, y_pred)
print(f"R-squared score on the entire dataset: {r2}")


In [ ]:
# Load the submission data
submission_df = pd.read_csv("submission.csv")

submission_X = submission_df.drop('UHI Index', axis=1)

In [ ]:
# prompt: join the lat longs on the average_of_average_minute_UHI feature from results

import pandas as pd
# Assuming 'result' DataFrame is already created as in the previous code

# Merge the 'average_of_average_minute_UHI' feature back into the original DataFrame
df = pd.merge(submission_X, result_df, on=['Latitude', 'Longitude'], how='left')

# Display the updated DataFrame
display(df)


In [ ]:
# prompt: load submission.csv and use the model to predict the UHI_Index

import pandas as pd

# Choose the best-performing model from cross-validation results
best_model_name = 'Random Forest'  # Replace with the actual best model
best_model = models[best_model_name]

# Retrain the best model on the entire training dataset (grouped_df)
best_model.fit(X, y)


# Make predictions on the submission data
submission_predictions = best_model.predict(submission_X)

# Create a new DataFrame for submission
submission_results = submission_X
submission_results['UHI_Index'] = submission_predictions

# Save the predictions to a CSV file
submission_results.to_csv('submission_predictions.csv', index=False)
